In [3]:
from pyspark.sql import SparkSession
from pyspark import SparkConf, SparkContext
from pyspark import StorageLevel
import random as r
import string

In [4]:
spark = (
        SparkSession.builder
        .appName("SparkCore Programs App")
        .master("local[*]")
        .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.1.0,"
                                        "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.0,"
                                        "org.apache.hudi:hudi-spark3.5-bundle_2.12:0.15.0,"
                                        "org.apache.hadoop:hadoop-aws:3.3.4,"
                                        "com.amazonaws:aws-java-sdk-bundle:1.12.262,"
                                        "software.amazon.awssdk:glue:2.25.26,"
                                        "software.amazon.awssdk:sts:2.25.26,"
                                        "software.amazon.awssdk:s3:2.25.26,"
                                        "software.amazon.awssdk:dynamodb:2.25.26,"
                                        "org.apache.iceberg:iceberg-aws-bundle:1.5.0,"
                                        "software.amazon.awssdk:url-connection-client:2.25.26")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension,"
                                        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,"
                                        "org.apache.spark.sql.hudi.HoodieSparkSessionExtension")
        .config("spark.hadoop.fs.s3.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        .config("spark.sql.catalog.glue_catalog", "org.apache.iceberg.spark.SparkCatalog")
        .config("spark.sql.catalog.glue_catalog.catalog-impl", "org.apache.iceberg.aws.glue.GlueCatalog")
        .config("spark.sql.catalog.glue_catalog.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
        .config("spark.sql.catalog.glue_catalog.warehouse", "s3a://shivchoudhury-datasets/warehouse/")
        .config("spark.sql.warehouse.dir", "s3a://shivchoudhury-datasets/warehouse/")
        .config("spark.sql.catalog.glue_catalog.client.region", "ap-south-1")
        .config("spark.sql.defaultCatalog", "spark_catalog")
        .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
        .config("spark.hadoop.fs.s3a.aws.credentials.provider", "com.amazonaws.auth.DefaultAWSCredentialsProviderChain")
        .config("spark.hadoop.fs.s3a.endpoint", "s3.ap-south-1.amazonaws.com")
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
        .config("hive.metastore.client.factory.class", "com.amazonaws.glue.catalog.metastore.AWSGlueDataCatalogHiveClientFactory")
        .enableHiveSupport()
        .getOrCreate()
    );

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
org.apache.hudi#hudi-spark3.5-bundle_2.12 added as a dependency
software.amazon.awssdk#glue added as a dependency
software.amazon.awssdk#sts added as a dependency
software.amazon.awssdk#s3 added as a dependency
software.amazon.awssdk#url-connection-client added as a dependency
software.amazon.awssdk#dynamodb added as a dependency
org.apache.iceberg#iceberg-aws-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-382cfbb7-7906-4188-93db-17fc1595a1e0;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.1.0 in central
	found io.delta#delta-storage;3.1.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.a

In [5]:
sc = spark.sparkContext

In [6]:
rdd1 = sc.parallelize([x for x in range(100)])
print(f"RDD : {rdd1.collect()}")
print(f"Repartitions: {rdd1.getNumPartitions()}")
print(f"Type: {type(rdd1)}")

RDD : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99]
Repartitions: 12
Type: <class 'pyspark.rdd.RDD'>


## reduce

In [7]:
rdd2 = sc.parallelize([r.randint(1, 50) for _ in range(100)], 4)
print(rdd2.collect(), end='\n\n')
rdd2 = rdd2.map(lambda x: (x, 1))
print(rdd2.collect(), end='\n\n')
rdd2 = rdd2.reduceByKey(lambda x, y: x + y)
print(rdd2.collect(), end='\n')

[48, 38, 42, 17, 47, 37, 14, 48, 43, 42, 50, 7, 12, 36, 36, 8, 10, 19, 3, 38, 3, 21, 24, 43, 46, 37, 16, 6, 6, 12, 50, 27, 50, 48, 29, 15, 33, 10, 34, 17, 16, 22, 38, 37, 32, 23, 37, 14, 34, 32, 1, 11, 31, 18, 35, 17, 20, 31, 7, 41, 5, 11, 29, 12, 36, 42, 22, 16, 27, 44, 7, 15, 45, 7, 3, 7, 2, 31, 9, 13, 19, 14, 15, 29, 31, 4, 49, 32, 19, 17, 2, 16, 22, 37, 2, 18, 22, 2, 30, 41]



[(48, 1), (38, 1), (42, 1), (17, 1), (47, 1), (37, 1), (14, 1), (48, 1), (43, 1), (42, 1), (50, 1), (7, 1), (12, 1), (36, 1), (36, 1), (8, 1), (10, 1), (19, 1), (3, 1), (38, 1), (3, 1), (21, 1), (24, 1), (43, 1), (46, 1), (37, 1), (16, 1), (6, 1), (6, 1), (12, 1), (50, 1), (27, 1), (50, 1), (48, 1), (29, 1), (15, 1), (33, 1), (10, 1), (34, 1), (17, 1), (16, 1), (22, 1), (38, 1), (37, 1), (32, 1), (23, 1), (37, 1), (14, 1), (34, 1), (32, 1), (1, 1), (11, 1), (31, 1), (18, 1), (35, 1), (17, 1), (20, 1), (31, 1), (7, 1), (41, 1), (5, 1), (11, 1), (29, 1), (12, 1), (36, 1), (42, 1), (22, 1), (16, 1), (27, 1), (44, 1), (7, 1), (15, 1), (45, 1), (7, 1), (3, 1), (7, 1), (2, 1), (31, 1), (9, 1), (13, 1), (19, 1), (14, 1), (15, 1), (29, 1), (31, 1), (4, 1), (49, 1), (32, 1), (19, 1), (17, 1), (2, 1), (16, 1), (22, 1), (37, 1), (2, 1), (18, 1), (22, 1), (2, 1), (30, 1), (41, 1)]

[(48, 3), (12, 3), (36, 3), (8, 1), (24, 1), (16, 4), (32, 3), (20, 1), (44, 1), (4, 1), (17, 4), (37, 5), (21, 1), (

In [8]:
rdd = sc.parallelize([r.randint(1, 50) for _ in range(100)], 2)
count = rdd.reduce(lambda x, y: x + y)
print(count)

2226


In [9]:
rdd = sc.parallelize([r.randint(1, 50) for _ in range(1000)], 4)
rdd = rdd.map(lambda x: (x, 1)).reduceByKeyLocally(lambda x, y: x + y)
print(rdd)

{34: 31, 41: 23, 18: 23, 3: 15, 24: 18, 38: 29, 39: 19, 46: 27, 28: 22, 45: 13, 5: 26, 43: 22, 47: 14, 40: 17, 1: 21, 37: 22, 20: 14, 29: 24, 35: 11, 26: 26, 33: 17, 14: 22, 30: 18, 12: 31, 25: 22, 31: 21, 9: 20, 16: 17, 7: 14, 11: 23, 23: 11, 21: 21, 6: 23, 49: 29, 2: 12, 36: 14, 19: 22, 27: 24, 10: 17, 48: 26, 32: 16, 50: 18, 22: 18, 44: 17, 13: 14, 17: 26, 42: 17, 15: 15, 8: 17, 4: 21}


In [10]:
rdd = sc.parallelize([string.ascii_lowercase[r.randint(0, 25)] for _ in range(1000)], 4)
res1 = rdd.reduce(lambda x, y: x + y)
print(res1, end='\n\n')
res2 = rdd.map(lambda x: (x, 1)).reduceByKey(lambda x, y: x + y)
print(res2.collect(), end='\n\n')
res3 = rdd.map(lambda x: (x, 1)).reduceByKeyLocally(lambda x, y: x + y)
print(res3)

ccwgpbmuwgbudpkrnnyrzmjxilevmysvsyzboesjgeuptpjafmnctsrxfwysakdmumoinvkpomcnsgdlxfpkmbkkojlptfbduokapyqqtyznippdznzpyyxyulzdipavoqobsummlctlcevevwtalhnjwagspaslwfxdipwbrpzieqnvfxkagzfitzgzuynuzdrhtfadvbxybgzjzkssqrriencavvqchzalfuttwyzztkczxvmvzmqepwrvlvjjdfdkklyrexvbkhjjakpcsdpkvttyjqhfojoocminbvdkdqyfozpeekbzosmydhmptnzatznqmmwwygovefhbjjorpbvijhumacovpnjkfqtskocuszswmhxsnfbafsmehryucdtxuhufshfxkunwcrsfemnwuwydfhbcoknkhfwzvimtchbaghwwozhwioezwqfvhdndidogwzwhomvaniwzupkhpddwqkgujhppyuzebrdvnizmcgmcxdjpkfhfaxyxcwjnzbbolxhgvyfxjqjukilfhmkyaerldoaiospcehyqqruwxicvipyzwsfmcnyxkrqbagspqelghdyaamormxpvzkisaidkkefhgbegklxhycazzphdovqrwxtvlbcnpowhwlmtuxghiahqysxxbahehvturupapmspzqjsnwvpufejamkxrdvhqvuyyyzacanchyhrcaiylsjxplbzqwvyghmkuxxdhkjnfxpxaeteripnuksvxejxyxmbghzvwocegipjfnbqvcmatluiuxkfwxbzspgjyhvphqtgneqghfbmqwfbxvxzagvqazlgwrkifnofrfodnvlkrtkqizlewbnpsjpyknnhitcawezghxlvpsjwhobohrtlxfzgkyauifnbjlniwblaebdqtbgtuoxbdatjftdgydinkicanzctegodmahffkirkrpgeluwbronzqepyignxsljovhskbmswhruezyz

## reparition

In [11]:
rdd = sc.parallelize([x for x in range(100)])
print(rdd.getNumPartitions())
rdd = rdd.repartition(10)
print(rdd.getNumPartitions())
print(rdd.glom().collect())

12
10
[[80, 81, 82, 83, 84, 85, 86, 87], [48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63], [], [88, 89, 90, 91, 92, 93, 94, 95, 96, 97], [8, 9, 10, 11, 12, 13, 14, 15, 32, 33, 34, 35, 36, 37, 38, 39, 98, 99], [], [40, 41, 42, 43, 44, 45, 46, 47, 64, 65, 66, 67, 68, 69, 70, 71], [], [24, 25, 26, 27, 28, 29, 30, 31], [0, 1, 2, 3, 4, 5, 6, 7, 16, 17, 18, 19, 20, 21, 22, 23, 72, 73, 74, 75, 76, 77, 78, 79]]


## coalesce

In [12]:
rdd = sc.parallelize([x for x in range(20)], 4)
print(rdd.getNumPartitions(), end='\n\n')
print(rdd.glom().collect(), end='\n\n')
r1 = rdd.repartition(3)
r2 = rdd.coalesce(3)
print(rdd.glom().collect(), end='\n\n')
print(r1.glom().collect(), end='\n\n')
print(r2.glom().collect(), end='\n\n')

4

[[0, 1, 2, 3, 4], [5, 6, 7, 8, 9], [10, 11, 12, 13, 14], [15, 16, 17, 18, 19]]

[[0, 1, 2, 3, 4], [5, 6, 7, 8, 9], [10, 11, 12, 13, 14], [15, 16, 17, 18, 19]]

[[0, 1, 2, 3, 4, 15, 16, 17, 18, 19], [5, 6, 7, 8, 9], [10, 11, 12, 13, 14]]

[[0, 1, 2, 3, 4], [5, 6, 7, 8, 9], [10, 11, 12, 13, 14, 15, 16, 17, 18, 19]]



## fold

In [13]:
r1 = sc.parallelize([1, 2, 3, 4, 5], 3)
print(r1.getNumPartitions(), end='\n\n')
print(r1.glom().collect(), end='\n\n')
print(r1.fold(2, lambda x, y: x + y), end='\n\n')

3

[[1], [2, 3], [4, 5]]

23



In [14]:
r1 = sc.parallelize([1, 2, 3, 4, 5], 3)
print(r1.getNumPartitions(), end='\n\n')
print(r1.glom().collect(), end='\n\n')
print(r1.fold(1, lambda x, y: x + y), end='\n\n')

3

[[1], [2, 3], [4, 5]]

19



In [15]:
r1 = sc.parallelize([string.ascii_lowercase[r.randint(0, 25)] for _ in range(20)], 3)
print(r1, end='\n\n')
r1 = r1.map(lambda x: (x, 1))
print(r1, end='\n\n')
print(r1.glom().collect(), end='\n\n')
print(r1.foldByKey(0, lambda x, y: x + y).glom().collect(), end='\n\n')
print(r1.foldByKey(1, lambda x, y: x + y).glom().collect(), end='\n\n')
print(r1.foldByKey(4, lambda x, y: x + y).glom().collect(), end='\n\n')

ParallelCollectionRDD[44] at readRDDFromFile at PythonRDD.scala:289

PythonRDD[45] at RDD at PythonRDD.scala:53

[[('y', 1), ('f', 1), ('m', 1), ('r', 1), ('x', 1), ('a', 1)], [('q', 1), ('n', 1), ('j', 1), ('w', 1), ('s', 1), ('m', 1)], [('d', 1), ('q', 1), ('j', 1), ('z', 1), ('m', 1), ('f', 1), ('j', 1), ('m', 1)]]

[[('x', 1), ('n', 1), ('w', 1), ('s', 1), ('d', 1), ('z', 1)], [('y', 1), ('q', 2)], [('f', 2), ('m', 4), ('r', 1), ('a', 1), ('j', 3)]]

[[('x', 2), ('n', 2), ('w', 2), ('s', 2), ('d', 2), ('z', 2)], [('y', 2), ('q', 4)], [('f', 4), ('m', 7), ('r', 2), ('a', 2), ('j', 5)]]

[[('x', 5), ('n', 5), ('w', 5), ('s', 5), ('d', 5), ('z', 5)], [('y', 5), ('q', 10)], [('f', 10), ('m', 16), ('r', 5), ('a', 5), ('j', 11)]]



## aggregate

In [16]:
seqOp = (lambda x, y: (x[0] + y, x[1] + 1))
combOp = (lambda x, y: (x[0] + y[0], x[1] + y[1]))

r1 = sc.parallelize([x for x in range(5)], 3)
print(r1.glom().collect(), end='\n\n')
print(r1.aggregate((0, 0), seqOp, combOp), end='\n\n')
print('--' * 20)
r2 = sc.parallelize([1, 2, 3, 4], 2)
print(r2.glom().collect(), end='\n\n')
print(r2.aggregate(1, lambda x, y: x + y, lambda x, y: (x + 1) * y))

[[0], [1, 2], [3, 4]]

(10, 5)

----------------------------------------
[[1, 2], [3, 4]]

72


In [17]:
r1 = sc.parallelize([r.randint(1, 5) for _ in range(10)], 2)
r1 = r1.map(lambda x: (x, 1))
print(r1.glom().collect(), end='\n\n')
r1 = r1.aggregateByKey(0, lambda x, y: x + y, lambda x, y: x + y)
print(r1.collect(), end='\n\n')

[[(4, 1), (1, 1), (1, 1), (4, 1), (2, 1)], [(4, 1), (3, 1), (4, 1), (2, 1), (5, 1)]]

[(4, 4), (2, 2), (1, 2), (3, 1), (5, 1)]



## groupBy

In [18]:
r1 = sc.parallelize([x for x in range(1, 26)], 3)
print(r1.glom().collect(), end='\n\n')
r1 = r1.groupBy(lambda x: x % 2)
print(r1.collect(), end='\n\n')
for x, y in r1.collect():
    print(x, list(y))

[[1, 2, 3, 4, 5, 6, 7, 8], [9, 10, 11, 12, 13, 14, 15, 16], [17, 18, 19, 20, 21, 22, 23, 24, 25]]

[(0, <pyspark.resultiterable.ResultIterable object at 0x7f2cd01fab30>), (1, <pyspark.resultiterable.ResultIterable object at 0x7f2ca13344c0>)]

0 [2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24]
1 [1, 3, 5, 7, 9, 11, 13, 15, 17, 19, 21, 23, 25]


In [19]:
r1 = sc.parallelize(['a', 'b', 'c', 'a', 'd', 'a', 'e', 'c', 'e', 'b', 'e'], 3)
r1 = r1.map(lambda x: (x, 1))
print(r1.glom().collect(), end='\n\n')
print(r1.groupByKey(2).collect(), end='\n\n')
res = r1.groupByKey(2).collect()
for x, y in res:
    print(x, list(y))

[[('a', 1), ('b', 1), ('c', 1)], [('a', 1), ('d', 1), ('a', 1)], [('e', 1), ('c', 1), ('e', 1), ('b', 1), ('e', 1)]]

[('b', <pyspark.resultiterable.ResultIterable object at 0x7f2ca11b4df0>), ('c', <pyspark.resultiterable.ResultIterable object at 0x7f2ca139a440>), ('d', <pyspark.resultiterable.ResultIterable object at 0x7f2ca139bca0>), ('a', <pyspark.resultiterable.ResultIterable object at 0x7f2ca139b340>), ('e', <pyspark.resultiterable.ResultIterable object at 0x7f2ca139a740>)]

b [1, 1]
c [1, 1]
d [1]
a [1, 1, 1]
e [1, 1, 1]


In [20]:
w = sc.parallelize([("a", 5), ("b", 6)])
x = sc.parallelize([("a", 1), ("b", 4)])
y = sc.parallelize([("a", 2)])
z = sc.parallelize([("b", 42)])
res = w.groupWith(x, y, z).collect()
print(res, end='\n\n')
for x, y in res:
    print(x, [list(i) for i in y])

[Stage 47:===============================================>        (41 + 7) / 48]

[('b', (<pyspark.resultiterable.ResultIterable object at 0x7f2ca11da860>, <pyspark.resultiterable.ResultIterable object at 0x7f2ca13a9720>, <pyspark.resultiterable.ResultIterable object at 0x7f2ca13a9f00>, <pyspark.resultiterable.ResultIterable object at 0x7f2ca13a9de0>)), ('a', (<pyspark.resultiterable.ResultIterable object at 0x7f2ca13a8ac0>, <pyspark.resultiterable.ResultIterable object at 0x7f2ca13a94e0>, <pyspark.resultiterable.ResultIterable object at 0x7f2ca13ab790>, <pyspark.resultiterable.ResultIterable object at 0x7f2ca13a88b0>))]

b [[6], [4], [], [42]]
a [[5], [1], [2], []]


In [21]:
a = sc.parallelize([1, 2, 3, 4, 5]).map(lambda x: (x, 1))
b = sc.parallelize([2, 3, 4, 5, 6]).map(lambda x: (x, 1))
c = sc.parallelize([3, 4, 5, 6, 7]).map(lambda x: (x, 1))
res = a.groupWith(b, c).collect()
print(res, end='\n\n')
for x, y in res:
    print(x, [list(i) for i in y])

[Stage 48:=================================================>      (32 + 4) / 36]

[(1, (<pyspark.resultiterable.ResultIterable object at 0x7f2ca11f0b20>, <pyspark.resultiterable.ResultIterable object at 0x7f2ca13abc40>, <pyspark.resultiterable.ResultIterable object at 0x7f2ca13aa4a0>)), (2, (<pyspark.resultiterable.ResultIterable object at 0x7f2ca13aa650>, <pyspark.resultiterable.ResultIterable object at 0x7f2ca13ab460>, <pyspark.resultiterable.ResultIterable object at 0x7f2ca13abbb0>)), (3, (<pyspark.resultiterable.ResultIterable object at 0x7f2ca13abbe0>, <pyspark.resultiterable.ResultIterable object at 0x7f2ca13ab940>, <pyspark.resultiterable.ResultIterable object at 0x7f2ca13ab8b0>)), (4, (<pyspark.resultiterable.ResultIterable object at 0x7f2ca13ab8e0>, <pyspark.resultiterable.ResultIterable object at 0x7f2ca13ab970>, <pyspark.resultiterable.ResultIterable object at 0x7f2ca13ab9a0>)), (5, (<pyspark.resultiterable.ResultIterable object at 0x7f2ca13ab9d0>, <pyspark.resultiterable.ResultIterable object at 0x7f2ca13ab670>, <pyspark.resultiterable.ResultIterable obj

In [22]:
a = sc.parallelize([("a", 1), ("b", 2)])
b = sc.parallelize([("b", 3), ("d", 4)])
res = a.cogroup(b)
print(res.collect(), end='\n\n')
for x, y in res.collect():
    print(x, [list(i) for i in y])

[('b', (<pyspark.resultiterable.ResultIterable object at 0x7f2ca11d9030>, <pyspark.resultiterable.ResultIterable object at 0x7f2ca13ab9d0>)), ('d', (<pyspark.resultiterable.ResultIterable object at 0x7f2ca13ab580>, <pyspark.resultiterable.ResultIterable object at 0x7f2ca13ab670>)), ('a', (<pyspark.resultiterable.ResultIterable object at 0x7f2ca13aa530>, <pyspark.resultiterable.ResultIterable object at 0x7f2ca13aa500>))]

b [[2], [3]]
d [[], [4]]
a [[1], []]


## count

In [23]:
a = sc.parallelize([x for x in range(100)], 3)
print(a.getNumPartitions(), end='\n\n')
print(a.glom().collect(), end='\n\n')
print(a.count(), end='\n\n')

b = sc.parallelize([r.randint(1, 5) for _ in range(30)], 3).map(lambda x: (x, 1))
print(b.glom().collect(), end='\n\n')
print(b.countByKey(), end='\n\n')
print(b.countByValue(), end='\n\n')

3

[[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32], [33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65], [66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99]]

100

[[(5, 1), (2, 1), (3, 1), (1, 1), (5, 1), (4, 1), (5, 1), (1, 1), (1, 1), (1, 1)], [(5, 1), (4, 1), (2, 1), (3, 1), (3, 1), (2, 1), (1, 1), (5, 1), (1, 1), (4, 1)], [(5, 1), (5, 1), (5, 1), (5, 1), (3, 1), (4, 1), (5, 1), (4, 1), (3, 1), (5, 1)]]

defaultdict(<class 'int'>, {5: 11, 2: 3, 3: 5, 1: 6, 4: 5})

defaultdict(<class 'int'>, {(5, 1): 11, (2, 1): 3, (3, 1): 5, (1, 1): 6, (4, 1): 5})



## filter

In [24]:
a = sc.parallelize([x for x in range(25)], 3)
print(a.getNumPartitions(), end='\n\n')
print(a.glom().collect(), end='\n\n')
a = a.filter(lambda x: x % 2 == 0)
print(a.collect(), end='\n\n')

b = sc.parallelize([r.randint(1, 5) for _ in range(35)], 3).map(lambda x: (x, 1)).reduceByKey(lambda x, y: x + y)
print(b.glom().collect(), end='\n\n')
b = b.filter(lambda x: x[0] % 2 == 0 and x[1] % 2 == 0)
print(b.collect(), end='\n\n')

3

[[0, 1, 2, 3, 4, 5, 6, 7], [8, 9, 10, 11, 12, 13, 14, 15], [16, 17, 18, 19, 20, 21, 22, 23, 24]]

[0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24]

[[(3, 4)], [(4, 6), (1, 9)], [(2, 12), (5, 4)]]

[(4, 6), (2, 12)]



## cache & persist

In [25]:
a = sc.parallelize([x for x in range(100)], 3)

In [26]:
a.glom().collect()

[[0,
  1,
  2,
  3,
  4,
  5,
  6,
  7,
  8,
  9,
  10,
  11,
  12,
  13,
  14,
  15,
  16,
  17,
  18,
  19,
  20,
  21,
  22,
  23,
  24,
  25,
  26,
  27,
  28,
  29,
  30,
  31,
  32],
 [33,
  34,
  35,
  36,
  37,
  38,
  39,
  40,
  41,
  42,
  43,
  44,
  45,
  46,
  47,
  48,
  49,
  50,
  51,
  52,
  53,
  54,
  55,
  56,
  57,
  58,
  59,
  60,
  61,
  62,
  63,
  64,
  65],
 [66,
  67,
  68,
  69,
  70,
  71,
  72,
  73,
  74,
  75,
  76,
  77,
  78,
  79,
  80,
  81,
  82,
  83,
  84,
  85,
  86,
  87,
  88,
  89,
  90,
  91,
  92,
  93,
  94,
  95,
  96,
  97,
  98,
  99]]

In [27]:
a.cache()

ParallelCollectionRDD[150] at readRDDFromFile at PythonRDD.scala:289

In [28]:
a.glom().collect()

[[0,
  1,
  2,
  3,
  4,
  5,
  6,
  7,
  8,
  9,
  10,
  11,
  12,
  13,
  14,
  15,
  16,
  17,
  18,
  19,
  20,
  21,
  22,
  23,
  24,
  25,
  26,
  27,
  28,
  29,
  30,
  31,
  32],
 [33,
  34,
  35,
  36,
  37,
  38,
  39,
  40,
  41,
  42,
  43,
  44,
  45,
  46,
  47,
  48,
  49,
  50,
  51,
  52,
  53,
  54,
  55,
  56,
  57,
  58,
  59,
  60,
  61,
  62,
  63,
  64,
  65],
 [66,
  67,
  68,
  69,
  70,
  71,
  72,
  73,
  74,
  75,
  76,
  77,
  78,
  79,
  80,
  81,
  82,
  83,
  84,
  85,
  86,
  87,
  88,
  89,
  90,
  91,
  92,
  93,
  94,
  95,
  96,
  97,
  98,
  99]]

In [29]:
a.getStorageLevel()

StorageLevel(False, True, False, False, 1)

In [30]:
a.unpersist()

ParallelCollectionRDD[150] at readRDDFromFile at PythonRDD.scala:289

In [31]:
a.getStorageLevel()

StorageLevel(False, False, False, False, 1)

In [35]:
a.persist(StorageLevel.MEMORY_AND_DISK)

ParallelCollectionRDD[150] at readRDDFromFile at PythonRDD.scala:289

In [36]:
a.count()

100

In [37]:
a.unpersist()

ParallelCollectionRDD[150] at readRDDFromFile at PythonRDD.scala:289

In [38]:
spark.stop()